# HW6

## 단계 1. 읽고 나누기

In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]
y = sales["total_sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
print("training :", len(X_train), " test :", len(X_test))

training : 803  test : 201


## 단계 2. model 셋의 test R²

In [2]:
def add_qp(frame):
    """수량 × 단가 열 qp 를 더한 새 DataFrame 을 돌려준다 (서버에서도 같은 이름으로 만든다)"""
    return frame.assign(qp=frame["quantity"] * frame["unit_price"])

X2_train = add_qp(X_train)
X2_test = add_qp(X_test)

lin = LinearRegression().fit(X_train, y_train)
lin2 = LinearRegression().fit(X2_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)

print("linear regression test R² :", round(lin.score(X_test, y_test), 3))
print("linear regression + qp test R² :", round(lin2.score(X2_test, y_test), 3))
print("decision tree (no limit) test R² :", round(tree.score(X_test, y_test), 3))

linear regression test R² : 0.862
linear regression + qp test R² : 1.0
decision tree (no limit) test R² : 0.994


## 단계 4. max_depth 별 잎 수, training R², test R²

In [3]:
rows = []
for dep in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=dep, random_state=0).fit(X_train, y_train)
    rows.append([str(dep), m.get_n_leaves(),
                 round(m.score(X_train, y_train), 3), round(m.score(X_test, y_test), 3)])
pd.DataFrame(rows, columns=["max_depth", "leaves", "training R²", "test R²"])

,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


## 단계 5. cross-validation (cv=5)

In [4]:
for dep in [3, 5, 8, 12, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=dep, random_state=0), X, y, cv=5)
    print(f"max_depth {str(dep):4s} mean {s.mean():.3f} std {s.std():.3f}")

max_depth 3    mean 0.845 std 0.019
max_depth 5    mean 0.961 std 0.003
max_depth 8    mean 0.992 std 0.000
max_depth 12   mean 0.995 std 0.001
max_depth None mean 0.995 std 0.001


## 단계 6. 고른 model 저장

In [5]:
QUANTITY, UNIT_PRICE = 40, 200000
order = pd.DataFrame([[QUANTITY, UNIT_PRICE]], columns=["quantity", "unit_price"])
tree8 = DecisionTreeRegressor(max_depth=8, random_state=0).fit(X_train, y_train)

print("linear + qp →", int(round(lin2.predict(add_qp(order))[0])))
print("tree (depth 8) →", int(round(tree8.predict(order)[0])))
print("quantity × price →", QUANTITY * UNIT_PRICE)

joblib.dump(lin2, "sales_model.pkl")

linear + qp → 8000000
tree (depth 8) → 7362301
quantity × price → 8000000


['sales_model.pkl']